# Temporal Partition Router for Large Models on Small Devices

A large model does not have to receive every request with the same temperature, memory window, and prompt strategy. A small routing layer can classify the request before model dispatch and choose a better execution profile.

This cookbook demonstrates a temporal router inspired by ALFA Guardian: **Yesterday / Today / Tomorrow**.

## Why Temporal Partitions?

- **Yesterday**: historical context, memory recall, retrospectives. Low temperature, larger memory window.
- **Today**: current task execution, debugging, active analysis. Balanced temperature and context.
- **Tomorrow**: planning, forecasting, strategy. Higher temperature, smaller focused context.

The point is not to make the model smarter by wishful prompting. The point is to send the model a better-shaped request.

In [ ]:
from dataclasses import asdict
from temporal_router import TemporalTagger, PartitionRouter

## Label Prompts Before Dispatch

The tagger assigns a partition, intent, domain, confidence score, and matched signals.

In [ ]:
tagger = TemporalTagger()

prompts = [
    "Remember what we changed yesterday in the Ollama setup.",
    "Sprawdz teraz build i napraw blad w pipeline.",
    "Build a roadmap for releasing this next week.",
]

labels = [tagger.label(prompt) for prompt in prompts]
[asdict(label) for label in labels]

## Route to a Model Profile

The router converts labels into model parameters and a message payload. This can be used with Grok, Ollama, or any OpenAI-compatible endpoint.

In [ ]:
router = PartitionRouter()

routed = [router.route(prompt, label, model_name="gpt-oss:120b") for prompt, label in zip(prompts, labels)]
[asdict(item) for item in routed]

## Optional: Call Ollama Safely

The same routed profile can be sent to a local Ollama model. On small devices, avoid loading multiple large models at once and keep `num_ctx` modest.

In [ ]:
# import requests
#
# def call_ollama(routed_request):
#     profile = routed_request.model_profile
#     body = {
#         "model": profile["model"],
#         "messages": routed_request.messages,
#         "stream": False,
#         "keep_alive": "30s",
#         "options": {
#             "temperature": profile["temperature"],
#             "num_predict": min(int(profile["num_predict"]), 512),
#             "num_ctx": 2048,
#         },
#     }
#     response = requests.post("http://127.0.0.1:11434/api/chat", json=body, timeout=180)
#     response.raise_for_status()
#     return response.json()
#
# call_ollama(routed[1])

## Optional: Use Grok Instead

For xAI/Grok, keep the same routing layer and swap only the adapter. The labels and partition settings remain provider-independent.

In [ ]:
# %pip install --quiet openai python-dotenv
# import os
# from dotenv import load_dotenv
# from openai import OpenAI
#
# load_dotenv()
# client = OpenAI(base_url="https://api.x.ai/v1", api_key=os.getenv("XAI_API_KEY"))
#
# def call_grok(routed_request):
#     profile = routed_request.model_profile
#     response = client.chat.completions.create(
#         model="grok-4",
#         messages=routed_request.messages,
#         temperature=float(profile["temperature"]),
#         max_tokens=min(int(profile["num_predict"]), 1024),
#     )
#     return response.choices[0].message.content
#
# call_grok(routed[2])

## Conclusion

Temporal routing is a lightweight control layer for large-model workflows. It helps a small machine or mixed-provider stack spend context and generation budget where it matters most.

The pattern composes naturally with pre-model filters and ALFA context snapshots: first decide whether a request is allowed, then label it, then route it to the right model profile.